# Item 15a — ajuste fixo das 8 configurações do encoder de atenção

Primeiro notebook da Phase 7 ([spec](../.specs/features/phase-7-atencao/spec.md), ATT-03). Ele
treina as **8 configurações** da grade fixada no
[ADR-0006](../docs/adr/0006-precompromisso-fase-atencao.md) antes de qualquer treino:
`heads ∈ {2, 4}` × dropout de atenção `∈ {0, 0,2}` × `lr ∈ {5e-4, 1e-3}`, com `hidden = 128`,
`encoder = "gatv2"`, no split `current`, seed 42. Escolhe a de menor `val_mse` e grava
`config_escolhida.json`, que os 12 treinos do item 15b usam sem mudar.

**Runtime: GPU (T4).** O notebook aborta logo depois do setup se não houver GPU.

Ordem, e por quê:

1. a suíte `tests/test_phase7_*.py` roda **como conferência** (como a célula 10 do item 06);
2. o **portão de vazamento** roda no grafo real: perturbar `days_together` e
   `avg_position_distance` no `edge_attr` não pode mudar nenhum embedding. Se mudar, o notebook
   para antes do primeiro treino;
3. cada configuração tem a contagem de parâmetros conferida (`check_param_budget`, 453.121 ± 10%)
   antes de treinar, e o treino é abortado fora da faixa;
4. a tabela de 8 linhas (`val_mse`, `n_params`) é gravada **antes** da escolha;
5. a escolha aplica a regra de empate da spec: diferença de `val_mse` ≤ 0,000003 (um desvio
   entre seeds da proposta SAGE no item 06) é empate, e vence a configuração com menos
   parâmetros. O notebook registra quando a regra decide.

Nenhuma avaliação em teste acontece aqui. **Retomada:** cada configuração grava seu checkpoint no
Drive assim que termina; uma sessão nova pula as que existem.

## 0. Ambiente — Colab (GPU) ou local

No Colab, clona o repositório (código + artefatos de dados versionados) e instala o PyG.
Ative a GPU em *Ambiente de execução → Alterar tipo de runtime → GPU*.
Repo privado: cole um PAT em `GITHUB_TOKEN`.

> Os grafos `hetero_full_current.pt` e `hetero_full_pre_pandemia.pt` precisam estar **commitados**
> antes de clonar: os CSVs brutos da rede de gêneros não são versionados, então o grafo não pode
> ser reconstruído aqui dentro.

In [ ]:
import sys, os, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    IS_COLAB = False

REPO_URL     = "https://github.com/cristianomendieta/music-influence-gnn.git"
REPO_BRANCH  = "main"
REPO_DIR     = "/content/music-influence-gnn"
GITHUB_TOKEN = ""   # repo privado: cole um PAT aqui OU defina a env GITHUB_TOKEN

DATA_FILES = [
    "data/processed/graph/hetero_full_current.pt",
    "data/processed/graph/hetero_full_pre_pandemia.pt",
    "data/processed/graph/node_id_map.json",
    "data/processed/timeseries.parquet",
]

def _clone(url):
    return subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, REPO_DIR])

if IS_COLAB:
    if Path(REPO_DIR, "pyproject.toml").exists():
        # O runtime sobrevive ao restart do kernel: um clone de sessão anterior ficaria
        # para trás em silêncio e o import viria do código velho. Sincroniza sempre.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{REPO_BRANCH}"], check=True)
    else:
        tok = GITHUB_TOKEN or os.environ.get("GITHUB_TOKEN", "")
        url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
        print(f"Clonando {REPO_URL} (branch {REPO_BRANCH})...")
        if _clone(url).returncode != 0:
            from getpass import getpass
            tok = getpass("Clone falhou (repo privado?). Cole um GitHub token (PAT): ")
            _clone(REPO_URL.replace("https://", f"https://{tok}@")).check_returncode()
    os.chdir(REPO_DIR)
    print("commit em uso:", subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
                                           capture_output=True, text=True).stdout.strip())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch-geometric", "pyarrow"], check=True)
    faltando = [f for f in DATA_FILES if not Path(REPO_DIR, f).exists()]
    print("✓ Colab pronto. cwd =", os.getcwd())
    if faltando:
        print("⚠️ FALTAM no repositório:", faltando)
else:
    print("Local — nada a clonar.")

In [ ]:
# Portão de GPU: este notebook treina/avalia modelos e só roda numa GPU do Colab (T4).
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Sem GPU: este notebook precisa de um runtime com GPU (T4). No Colab: Ambiente de "
        "execução → Alterar tipo de runtime → GPU, e rode de novo. Abortado antes de qualquer "
        "treino ou avaliação.")
DEVICE = "cuda"
print("DEVICE =", DEVICE, "|", torch.cuda.get_device_name(0))

In [ ]:
import json, time, hashlib, itertools, dataclasses
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_anchor = Path(globals().get("__vsc_ipynb_file__", os.getcwd()))
if not _anchor.exists():
    _anchor = Path(os.getcwd())
ROOT = _anchor if _anchor.is_dir() else _anchor.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), f"raiz do projeto não encontrada a partir de {_anchor}"
sys.path.insert(0, str(ROOT / "src"))

%matplotlib inline
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

from music_diffusion_gnn.evaluation.veredito_atencao import N_CONFIGS_ATTENTION, TIE_VAL_MSE
from music_diffusion_gnn.graph.build import graph_path
from music_diffusion_gnn.models.diffusion_gnn import MusicDiffusionGNN
from music_diffusion_gnn.training.dataset import (
    aggregate_weekly, build_pop_bank, build_samples, temporal_split,
)
from music_diffusion_gnn.training.trainer import (
    PARAM_BUDGET_REF, Config, check_param_budget, checkpoint_dict, train_one,
)

GRAPH_DIR = ROOT / "data" / "processed" / "graph"
NMAP_PATH = GRAPH_DIR / "node_id_map.json"
TS_PATH   = ROOT / "data" / "processed" / "timeseries.parquet"
print("ROOT =", ROOT, "| DEVICE =", DEVICE)

In [ ]:
# Artefatos DURÁVEIS no Drive (sobrevivem à desconexão do Colab)
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = Path("/content/drive/MyDrive/music-influence-gnn/atencao/ajuste")
else:
    OUT = ROOT / "results" / "atencao" / "ajuste"
OUT.mkdir(parents=True, exist_ok=True)
CONFIG_JSON = OUT.parent / "config_escolhida.json"
print("artefatos →", OUT)

## 1. A suíte da Phase 7, como conferência

Os testes de `tests/test_phase7_*.py` cobrem o encoder num grafo sintético: atributos causais,
invariância às colunas brutas, paridade com o sem_grafo sem arestas, pesos que somam 1, orçamento
de parâmetros e checkpoint. Aqui eles rodam como **conferência**, não como portão: um problema de
ambiente do Colab não pode derrubar a sessão. O portão que vale é o da seção 3, sobre o grafo
real, e nenhuma célula antes desta carrega o grafo real.

In [ ]:
testes_p7 = sorted(str(p.relative_to(ROOT)) for p in (ROOT / "tests").glob("test_phase7_*.py"))
print("arquivos:", testes_p7)
r_pytest = subprocess.run([sys.executable, "-m", "pytest", "-q", "--no-header", *testes_p7],
                          capture_output=True, text=True, cwd=ROOT)
print((r_pytest.stdout or r_pytest.stderr).strip()[-1500:])
if r_pytest.returncode != 0:
    print(f"\n⚠️ pytest saiu com {r_pytest.returncode}. Leia a saída acima antes de seguir; "
          "o portão de vazamento da seção 3 roda de qualquer forma.")

## 2. A grade fixa e os dados de `current`

A grade é a do ADR-0006, sem nada a mais. `W = 12`, `layers = 3` e `dropout = 0,2` são os da
proposta SAGE; `max_cotraj_edges = 30_000` e 100 épocas (`patience = 10`) são os do protocolo da escada (ATT-04,
AC6), iguais aqui e no item 15b.

In [ ]:
REGIME, SEED = "current", 42
W, HIDDEN, LAYERS, DROPOUT = 12, 128, 3, 0.2
MAX_EPOCAS = 100                    # ADR-0006 e spec (AC6 do treino): 100 épocas, patience 10 da escada
MAX_COTRAJ_TREINO = 30_000
TEMPO_LIMITE_MIN = 150              # corta o laço antes da sessão do Colab cair

GRADE = [
    Config(W=W, hidden=HIDDEN, layers=LAYERS, lr=lr, dropout=DROPOUT, seed=SEED,
           max_epochs=MAX_EPOCAS, max_cotraj_edges=MAX_COTRAJ_TREINO,
           encoder="gatv2", heads=heads, attn_dropout=ad)
    for heads, ad, lr in itertools.product((2, 4), (0.0, 0.2), (5e-4, 1e-3))
]
assert len(GRADE) == N_CONFIGS_ATTENTION, (len(GRADE), N_CONFIGS_ATTENTION)
assert len({str(c) for c in GRADE}) == len(GRADE), "nomes de configuração repetidos"
for i, c in enumerate(GRADE, 1):
    print(f"  {i}. {c}")

In [ ]:
ts = pd.read_parquet(TS_PATH)
weekly = aggregate_weekly(ts)
first_seen = weekly.groupby(["song_id", "chart"], observed=True)["week"].min().to_dict()

g_path = graph_path(REGIME, GRAPH_DIR)
g = torch.load(g_path, weights_only=False)
GRAPH_VERSION = f"{g_path.name}@{hashlib.sha256(g_path.read_bytes()).hexdigest()[:12]}"
splits_df = temporal_split(weekly, regime=REGIME)
pop_bank = build_pop_bank(weekly, NMAP_PATH, n_music=g["music"].num_nodes)
samples = {
    nome: build_samples(splits_df[nome], W=W, node_id_map_path=NMAP_PATH, first_seen=first_seen)
    for nome in ("train", "val")
}
print(f"{REGIME}: " + " ".join(f"{k}={len(v):,}" for k, v in samples.items())
      + f" | grafo {GRAPH_VERSION}")

## 3. Portão de vazamento no grafo real (ADR-0006)

`days_together` e `avg_position_distance` (colunas 0 e 1 do `edge_attr` da cotrajetória) são
somadas sobre o período inteiro e vazariam o futuro para o snapshot da semana `w`. O encoder só
pode ler `chart` e a idade da aresta. Aqui um modelo `gatv2` (pesos iniciais, modo `eval`) codifica
semanas do treino sobre o grafo real e sobre uma cópia com as duas colunas trocadas por ruído
grande; os embeddings precisam ser **idênticos**. Se não forem, o notebook para aqui.

O grafo é usado inteiro (`max_cotraj_edges=None`) para que a subamostragem aleatória das arestas
não entre na comparação.

In [ ]:
COTRAJ = ("music", "cotrajectory", "music")
g_pert = g.clone()
ea = g_pert[COTRAJ].edge_attr.clone()
gen = torch.Generator().manual_seed(0)
ea[:, 0] = torch.randn(ea.shape[0], generator=gen) * 1e3
ea[:, 1] = torch.randn(ea.shape[0], generator=gen) * 1e3
g_pert[COTRAJ].edge_attr = ea

semanas_portao = sorted({s.target_week for s in samples["train"]})[::40][:4]
torch.manual_seed(SEED)
m_portao = MusicDiffusionGNN(g.metadata(), hidden=HIDDEN, layers=LAYERS, dropout=DROPOUT,
                             pop_bank=pop_bank, encoder="gatv2", heads=4,
                             attn_dropout=0.2).to(DEVICE).eval()
with torch.no_grad():
    bank_real = m_portao.encode_weeks(g.to(DEVICE), semanas_portao, max_cotraj_edges=None)
    bank_pert = m_portao.encode_weeks(g_pert.to(DEVICE), semanas_portao, max_cotraj_edges=None)
difs = {w: (bank_real[w] - bank_pert[w]).abs().max().item() for w in semanas_portao}
print("semanas:", semanas_portao)
print("maior diferença por semana:", difs)
if not all(torch.equal(bank_real[w], bank_pert[w]) for w in semanas_portao):
    raise RuntimeError(
        "PORTÃO DE VAZAMENTO FALHOU: o encoder lê days_together/avg_position_distance no grafo "
        "real. Nenhum treino pode ser interpretado; corrija o encoder antes de seguir.")
print("✓ portão de vazamento: embeddings idênticos com as colunas brutas perturbadas")
del m_portao, g_pert, bank_real, bank_pert
torch.cuda.empty_cache()

## 4. O laço do ajuste

Para cada configuração: um modelo é instanciado e aquecido com o forward de uma semana (os
parâmetros do `GATv2Conv` são lazy), a contagem passa por `check_param_budget` (fora de
407.808 a 498.433 o treino **não começa**: a exceção interrompe a célula) e só então `train_one`
treina. O checkpoint, com os metadados de `checkpoint_dict`, vai para o Drive assim que o treino
termina. Rode a célula quantas vezes precisar.

In [ ]:
def parametros_aquecidos(cfg: Config) -> int:
    """Conta os parâmetros do modelo que train_one vai construir, depois do forward de uma semana."""
    m = MusicDiffusionGNN(g.metadata(), hidden=cfg.hidden, layers=cfg.layers, dropout=cfg.dropout,
                          pop_bank=pop_bank, encoder=cfg.encoder, heads=cfg.heads,
                          attn_dropout=cfg.attn_dropout).to(DEVICE)
    with torch.no_grad():
        m.encode_weeks(g.to(DEVICE), [semanas_portao[0]], max_cotraj_edges=cfg.max_cotraj_edges)
    n = m.count_params()
    del m
    torch.cuda.empty_cache()
    return n

def ckpt_path(cfg: Config) -> Path:
    return OUT / f"ajuste_{cfg}.pt"

t_inicio = time.time()
for cfg in GRADE:
    alvo = ckpt_path(cfg)
    if alvo.exists():
        r = torch.load(alvo, map_location="cpu", weights_only=False)
        print(f"{cfg}: pronto (val_mse={r['val_mse']:.6f}) — pulando")
        continue
    decorrido = (time.time() - t_inicio) / 60
    if decorrido > TEMPO_LIMITE_MIN:
        print(f"\nlimite de {TEMPO_LIMITE_MIN} min atingido ({decorrido:.0f} min). Rode a célula "
              "de novo numa sessão nova: as configurações prontas serão puladas.")
        break

    n_aquecido = parametros_aquecidos(cfg)
    check_param_budget(n_aquecido)          # aborta o treino fora de 453.121 ± 10% (AC6)
    print(f"\n▶ {cfg} — {n_aquecido:,} parâmetros ({n_aquecido / PARAM_BUDGET_REF - 1:+.1%} "
          f"contra {PARAM_BUDGET_REF:,})")
    r = train_one(cfg, {"train": samples["train"], "val": samples["val"]}, g,
                  device=DEVICE, pop_bank=pop_bank)
    assert r.n_params == n_aquecido, (r.n_params, n_aquecido)
    ck = checkpoint_dict(cfg, r, variante="atencao", split_regime=REGIME, graph_seed=None,
                         graph_version=GRAPH_VERSION)
    ck["elapsed_sec"] = r.elapsed_sec
    torch.save(ck, alvo)
    print(f"  val_mse={r.val_mse:.6f}  épocas={len(r.val_curve)}  t={r.elapsed_sec/60:.1f}min "
          f"→ {alvo.name}")
    torch.cuda.empty_cache()

print("\nfeitas:", sum(ckpt_path(c).exists() for c in GRADE), f"de {len(GRADE)}")

## 5. A tabela das 8 configurações, gravada antes da escolha

A tabela é escrita em `ajuste_8_configs.parquet` **antes** de qualquer escolha, com `val_mse`,
`n_params` e o tempo de cada treino. `check_param_budget` roda de novo sobre cada linha. Se alguma
configuração ainda não treinou, a célula para: a escolha só vale sobre as 8.

O tempo médio por treino serve para recalcular o prazo dos 12 treinos do item 15b antes de
lançá-los (ADR-0006: matriz até 2026-11-27).

In [ ]:
faltam = [str(c) for c in GRADE if not ckpt_path(c).exists()]
if faltam:
    raise RuntimeError(f"{len(faltam)} configuração(ões) sem treino: {faltam}. Rode a seção 4 de novo.")

linhas = []
for cfg in GRADE:
    ck = torch.load(ckpt_path(cfg), map_location="cpu", weights_only=False)
    check_param_budget(ck["n_params"])
    linhas.append({"config_str": ck["config_str"], "heads": ck["heads"],
                   "attn_dropout": ck["attn_dropout"], "lr": ck["lr"], "val_mse": ck["val_mse"],
                   "n_params": ck["n_params"], "epocas": len(ck["val_curve"]),
                   "elapsed_min": ck["elapsed_sec"] / 60, "graph_version": ck["graph_version"]})
ajuste = pd.DataFrame(linhas)
assert len(ajuste) == N_CONFIGS_ATTENTION
assert ajuste.graph_version.nunique() == 1, "configurações treinadas sobre grafos diferentes"
ajuste.to_parquet(OUT / "ajuste_8_configs.parquet", index=False)
display(ajuste.sort_values("val_mse"))

t_medio = ajuste.elapsed_min.mean()
print(f"tempo médio por treino: {t_medio:.1f} min → 12 treinos do item 15b ≈ {12 * t_medio / 60:.1f} h "
      "de GPU (a referência do SAGE é ~7 min por treino)")

## 6. A escolha e a regra de empate

Menor `val_mse` vence. Toda configuração cuja diferença para a melhor é ≤ 0,000003
(`TIE_VAL_MSE`, importado do módulo do veredito) conta como **empatada** com ela; entre as
empatadas vence a de **menos parâmetros**, e entre empatadas com o mesmo número de parâmetros, a de
menor `val_mse`. A célula registra se houve empate e se a regra mudou a escolha.

`config_escolhida.json` guarda a configuração e o hash dela. Ele é criado uma vez: se já existir,
esta célula só confere que a escolha é a mesma e para se não for. O item 15b aborta se o arquivo
faltar ou se o hash não bater (ATT-03, AC2).

In [ ]:
melhor = ajuste.loc[ajuste.val_mse.idxmin()]
empatadas = ajuste[ajuste.val_mse - melhor.val_mse <= TIE_VAL_MSE]
escolhida = empatadas.sort_values(["n_params", "val_mse"]).iloc[0]
empate = len(empatadas) > 1
regra_decidiu = escolhida.config_str != melhor.config_str
print(f"menor val_mse: {melhor.config_str} ({melhor.val_mse:.6f}, {melhor.n_params:,} parâmetros)")
print(f"empatadas (Δval_mse ≤ {TIE_VAL_MSE}): {list(empatadas.config_str)}")
print(f"escolhida: {escolhida.config_str}"
      + (" — a REGRA DE EMPATE decidiu (menos parâmetros)" if regra_decidiu else ""))

cfg_escolhida = next(c for c in GRADE if str(c) == escolhida.config_str)
campos = {k: v for k, v in dataclasses.asdict(cfg_escolhida).items() if k != "seed"}
hash_cfg = hashlib.sha256(json.dumps(campos, sort_keys=True).encode()).hexdigest()
registro = {
    "config": campos, "config_str": escolhida.config_str, "hash": hash_cfg,
    "val_mse": float(escolhida.val_mse), "n_params": int(escolhida.n_params),
    "selecao": {"regime": REGIME, "seed": SEED, "graph_version": GRAPH_VERSION,
                "criterio": "menor val_mse; empate se Δ ≤ TIE_VAL_MSE, vence menos parâmetros",
                "tie_val_mse": TIE_VAL_MSE, "melhor_val_mse": melhor.config_str,
                "empatadas": list(empatadas.config_str), "empate": bool(empate),
                "regra_de_empate_decidiu": bool(regra_decidiu)},
    "criado_em": time.strftime("%Y-%m-%d %H:%M:%S"),
}
if CONFIG_JSON.exists():
    antigo = json.loads(CONFIG_JSON.read_text())
    if antigo["hash"] != hash_cfg:
        raise RuntimeError(
            f"{CONFIG_JSON} já existe com outra configuração ({antigo['config_str']}). A escolha "
            "não muda depois de criada (ADR-0006); investigue antes de apagar qualquer coisa.")
    print(f"\n{CONFIG_JSON.name} já existe e confere (hash {hash_cfg[:12]}).")
else:
    CONFIG_JSON.write_text(json.dumps(registro, indent=2, ensure_ascii=False))
    print(f"\n✓ {CONFIG_JSON} gravado (hash {hash_cfg[:12]})")

## 7. O que levar de volta

Do Drive (`atencao/`):

- `ajuste/ajuste_{config}.pt` — os 8 checkpoints do ajuste, com os metadados de `checkpoint_dict`;
- `ajuste/ajuste_8_configs.parquet` — a tabela de `val_mse` e parâmetros, gravada antes da escolha;
- `config_escolhida.json` — a configuração e o hash que o item 15b exige.

Antes de lançar o item 15b: recalcular o prazo com o tempo médio por treino medido acima.